In [ ]:
import random
import itertools
import json
import os
import asyncio
import pandas as pd
from tqdm.asyncio import tqdm_asyncio
from tqdm.notebook import tqdm
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_string, encode_image
from vpei.common_variables import *
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_utils import (
    print_comparative_experiment_results,
    generate_two_different_full_names_initials,
    derive_political_pole_of_model_response,
    derive_positional_response_of_model
)
from local_variables import PAINTING_STYLES

In [ ]:
experiment_name = "art"
system_prompt = EXPERIMENTS[experiment_name]["comparative_experiment_without_ground_truth"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["comparative_experiment_without_ground_truth"]["user_prompt_template"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
# Test single request with two paintings
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})

painting_style_1 = "abstract art"
painting_style_2 = "realism"
image_path_1 = f"./data/{painting_style_1}/0.png"
image_path_2 = f"./data/{painting_style_2}/0.png"

base64_image_1 = encode_image(image_path_1)
base64_image_2 = encode_image(image_path_2)

name_1 = "J.S."
name_2 = "M.J."
political_attitude_1 = "progressive"
political_attitude_2 = "conservative"

user_prompt_text = user_prompt_template.format(
    name_1=name_1, political_attitude_1=political_attitude_1,
    name_2=name_2, political_attitude_2=political_attitude_2
)

user_prompt = [
    {"type": "text", "text": "Painting A:"},
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image_1}"}},
    {"type": "text", "text": "Painting B:"},
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image_2}"}},
    {"type": "text", "text": user_prompt_text}
]

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

response = make_llm_request(model_name, messages, **model_kwargs)
print("Response:", response)

In [ ]:
async def _wrap_awaitable(index, awaitable):
    try:
        result = await awaitable
        return index, result
    except Exception as e:
        print(f"Error in task {index}: {e}")
        return index, None


async def carry_out_art_comparative_experiment_without_ground_truth(
    models, n, system_prompt, user_prompt_template, data_path,
    custom_model_kwargs={}, path_to_save_model_outputs="./comparative_experiment_without_ground_truth", random_seed=42, **kwargs
):
    _POLITICAL_ATTITUDES_CATEGORIES = kwargs.get("POLITICAL_ATTITUDES_CATEGORIES", POLITICAL_ATTITUDES_CATEGORIES)

    # Collect all available paintings
    all_paintings = []
    for painting_style in PAINTING_STYLES:
        style_path = f"{data_path}/{painting_style}"
        if not os.path.exists(style_path):
            continue
        available_images = [f for f in os.listdir(style_path) if f.endswith('.png')]
        for img in available_images:
            all_paintings.append((painting_style, img))

    async def run_model(model_name, position=0):
        random.seed(random_seed)
        # For logging to CSV only — adapt_model_kwargs_for_model is also called inside make_llm_request
        model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
        tasks = []

        for i in range(n//4):
            if len(all_paintings) < 2:
                print("Not enough paintings to compare")
                continue

            painting_A, painting_B = random.sample(all_paintings, 2)
            style_A, img_A = painting_A
            style_B, img_B = painting_B

            image_path_A = f"{data_path}/{style_A}/{img_A}"
            image_path_B = f"{data_path}/{style_B}/{img_B}"

            base64_image_A = encode_image(image_path_A)
            base64_image_B = encode_image(image_path_B)

            name_1, name_2 = generate_two_different_full_names_initials()

            for img_order in [(painting_A, painting_B, base64_image_A, base64_image_B),
                              (painting_B, painting_A, base64_image_B, base64_image_A)]:
                painting_1, painting_2, base64_1, base64_2 = img_order
                style_1, img_file_1 = painting_1
                style_2, img_file_2 = painting_2

                # Choose a political_attitude_category randomly
                political_attitude_category = random.choice(list(_POLITICAL_ATTITUDES_CATEGORIES.keys()))
                for political_pole_1, political_pole_2 in list(itertools.permutations(["right", "left"])):
                    political_attitude_1 = _POLITICAL_ATTITUDES_CATEGORIES[political_attitude_category][political_pole_1]
                    political_attitude_2 = _POLITICAL_ATTITUDES_CATEGORIES[political_attitude_category][political_pole_2]

                    user_prompt_text = user_prompt_template.format(
                        name_1=name_1, political_attitude_1=political_attitude_1,
                        name_2=name_2, political_attitude_2=political_attitude_2
                    )

                    user_prompt = [
                        {"type": "text", "text": "Painting A:"},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_1}"}},
                        {"type": "text", "text": "Painting B:"},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_2}"}},
                        {"type": "text", "text": user_prompt_text}
                    ]

                    payload = {
                        "model_name": model_name,
                        "system_prompt": system_prompt,
                        "user_prompt_text": user_prompt_text,
                        "model_kwargs": json.dumps(model_kwargs),
                        "painting_style_1": style_1,
                        "painting_style_2": style_2,
                        "image_file_1": img_file_1,
                        "image_file_2": img_file_2,
                        "name_1": name_1,
                        "name_2": name_2,
                        "political_attitude_category": political_attitude_category,
                        "political_pole_1": political_pole_1,
                        "political_pole_2": political_pole_2,
                        "political_attitude_1": political_attitude_1,
                        "political_attitude_2": political_attitude_2,
                    }

                    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
                    tasks.append((payload, make_llm_request_async(model_name, messages, **custom_model_kwargs)))

        # Run tasks with progress bar
        results = [None] * len(tasks)
        coros = [_wrap_awaitable(i, t[-1]) for i, t in enumerate(tasks)]
        for fut in tqdm(asyncio.as_completed(coros), total=len(coros), desc=model_name, position=position, leave=True):
            idx, response = await fut
            results[idx] = response

        payloads = []
        for idx, (payload, _) in enumerate(tasks):
            try:
                response = results[idx]
                payload['model_response_raw'] = response
                payload['model_response'] = extract_string(response)
                payload = derive_political_pole_of_model_response(payload)
                payload = derive_positional_response_of_model(payload)
            except Exception as e:
                print(f"Error processing response for task {idx} in model {model_name}: {e}")
                payload['model_response_raw'] = results[idx] if idx < len(results) else None
                payload['model_response'] = None
                payload['model_response_political_attitude'] = None
                payload['model_response_pole'] = None
                payload['model_response_position'] = None
            payloads.append(payload)

        df_results = pd.DataFrame(payloads)
        save_model_experimental_results_to_csv(df_results, path_to_save_model_outputs, model_name, model_kwargs=model_kwargs)
        return payloads

    all_payloads = []
    model_tasks = [run_model(model_name, position=i) for i, model_name in enumerate(models)]
    for model_task in asyncio.as_completed(model_tasks):
        payloads = await model_task
        all_payloads.extend(payloads)
    return all_payloads

In [ ]:
models = ["gpt-5-mini"]


n = 10  
custom_model_kwargs = {}
random_seed = 42
path_to_save_model_outputs = "./comparative_experiment_without_ground_truth"
data_path = "./data"


In [ ]:
payloads = await carry_out_art_comparative_experiment_without_ground_truth(
    models=models,
    n=n,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
    data_path=data_path,
    custom_model_kwargs=custom_model_kwargs,
    path_to_save_model_outputs=path_to_save_model_outputs,
    random_seed=random_seed
)

print_comparative_experiment_results(payloads, models)